Load our processed data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import os
import gc

PROJECT_DIR = "/content/drive/MyDrive/GC210_Retail_Forecasting"
PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"

df = pd.read_parquet(
    f"{PROCESSED_DIR}/m5_development_300_series.parquet"
)

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    ["id", "date"]
).reset_index(drop=True)

print("Shape:", df.shape)
print("Date:", df["date"].min(), "→", df["date"].max())

Shape: (573900, 24)
Date: 2011-01-29 00:00:00 → 2016-04-24 00:00:00


Create lag features

In [ ]:
LAG_DAYS = [1, 7, 14, 28]

for lag in LAG_DAYS:
    df[f"lag_{lag}"] = (
        df.groupby("id")["sales"]
        .shift(lag)
    )

print("Lag features created.")

df[
    ["id", "date", "sales",
     "lag_1", "lag_7", "lag_14", "lag_28"]
].head(35)

Lag features created.


,id,date,sales,lag_1,lag_7,lag_14,lag_28
0,FOODS_1_005_TX_3_validation,2011-01-29,2,NaN,NaN,NaN,NaN
1,FOODS_1_005_TX_3_validation,2011-01-30,3,2.0,NaN,NaN,NaN
2,FOODS_1_005_TX_3_validation,2011-01-31,3,3.0,NaN,NaN,NaN
3,FOODS_1_005_TX_3_validation,2011-02-01,2,3.0,NaN,NaN,NaN
4,FOODS_1_005_TX_3_validation,2011-02-02,0,2.0,NaN,NaN,NaN
5,FOODS_1_005_TX_3_validation,2011-02-03,2,0.0,NaN,NaN,NaN
6,FOODS_1_005_TX_3_validation,2011-02-04,2,2.0,NaN,NaN,NaN
7,FOODS_1_005_TX_3_validation,2011-02-05,3,2.0,2.0,NaN,NaN
8,FOODS_1_005_TX_3_validation,2011-02-06,2,3.0,3.0,NaN,NaN
9,FOODS_1_005_TX_3_validation,2011-02-07,2,2.0,3.0,NaN,NaN


Create leakage-safe rolling features

In [ ]:
ROLLING_WINDOWS = [7, 14, 28]

for window in ROLLING_WINDOWS:
    df[f"rolling_mean_{window}"] = (
        df.groupby("id")["sales"]
        .transform(
            lambda x: x.shift(1).rolling(
                window=window,
                min_periods=window
            ).mean()
        )
    )

print("Rolling mean features created.")

Rolling mean features created.


Rolling standard deviation

In [ ]:
for window in [7, 28]:
    df[f"rolling_std_{window}"] = (
        df.groupby("id")["sales"]
        .transform(
            lambda x: x.shift(1).rolling(
                window=window,
                min_periods=window
            ).std()
        )
    )

print("Rolling standard deviation features created.")

Rolling standard deviation features created.


Create demand trend features

In [ ]:
df["demand_ratio_7_28"] = (
    df["rolling_mean_7"] /
    (df["rolling_mean_28"] + 1e-6)
)

df["demand_change_7"] = (
    df["rolling_mean_7"] -
    df["rolling_mean_28"]
)

print("Demand trend features created.")

Demand trend features created.


Price features

In [ ]:
df["price_lag_1"] = (
    df.groupby("id")["sell_price"]
    .shift(1)
)

df["price_change"] = (
    df["sell_price"] -
    df["price_lag_1"]
)

df["price_change_pct"] = (
    df["price_change"] /
    (df["price_lag_1"] + 1e-6)
)

print("Price features created.")

Price features created.


Price volatility

In [ ]:
df["price_volatility_7"] = (
    df.groupby("id")["sell_price"]
    .transform(
        lambda x: x.shift(1).rolling(
            7,
            min_periods=7
        ).std()
    )
)

print("Price volatility feature created.")

Price volatility feature created.


Calendar features

In [ ]:
df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_month"] = df["date"].dt.day
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["year"] = df["date"].dt.year

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

print("Calendar features created.")

Calendar features created.


Cyclical encoding

In [ ]:
df["dow_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["dow_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)

df["month_sin"] = np.sin(
    2 * np.pi * df["month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["month"] / 12
)

df["week_sin"] = np.sin(
    2 * np.pi * df["week_of_year"] / 52
)

df["week_cos"] = np.cos(
    2 * np.pi * df["week_of_year"] / 52
)

print("Cyclical features created.")

Cyclical features created.


Event features

In [ ]:
df["has_event_1"] = (
    df["event_name_1"] != "None"
).astype(int)

df["has_event_2"] = (
    df["event_name_2"] != "None"
).astype(int)

df["has_event"] = (
    (df["has_event_1"] == 1) |
    (df["has_event_2"] == 1)
).astype(int)

print(
    df[
        ["date",
         "event_name_1",
         "event_name_2",
         "has_event"]
    ].head(20)
)

         date   event_name_1 event_name_2  has_event
0  2011-01-29           None         None          0
1  2011-01-30           None         None          0
2  2011-01-31           None         None          0
3  2011-02-01           None         None          0
4  2011-02-02           None         None          0
5  2011-02-03           None         None          0
6  2011-02-04           None         None          0
7  2011-02-05           None         None          0
8  2011-02-06      SuperBowl         None          1
9  2011-02-07           None         None          0
10 2011-02-08           None         None          0
11 2011-02-09           None         None          0
12 2011-02-10           None         None          0
13 2011-02-11           None         None          0
14 2011-02-12           None         None          0
15 2011-02-13           None         None          0
16 2011-02-14  ValentinesDay         None          1
17 2011-02-15           None         None     

SNAP features

In [ ]:
snap_columns = [
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

for col in snap_columns:
    df[col] = df[col].fillna(0).astype(int)

print(df[snap_columns].head())

   snap_CA  snap_TX  snap_WI
0        0        0        0
1        0        0        0
2        0        0        0
3        1        1        0
4        1        0        1


Create an overall SNAP indicator

In [ ]:
df["snap_any"] = (
    (df["snap_CA"] == 1) |
    (df["snap_TX"] == 1) |
    (df["snap_WI"] == 1)
).astype(int)

print(
    df["snap_any"].value_counts()
)

snap_any
0    290400
1    283500
Name: count, dtype: int64


Encode categorical variables

In [ ]:
categorical_columns = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

for col in categorical_columns:
    df[col] = df[col].astype("category")

print(
    df[categorical_columns].dtypes
)

item_id     category
dept_id     category
cat_id      category
store_id    category
state_id    category
dtype: object


Convert categories to integer codes

In [ ]:
for col in categorical_columns:
    df[f"{col}_code"] = (
        df[col].cat.codes
    )

print(
    df[
        [f"{col}_code" for col in categorical_columns]
    ].head()
)

   item_id_code  dept_id_code  cat_id_code  store_id_code  state_id_code
0             0             0            0              6              1
1             0             0            0              6              1
2             0             0            0              6              1
3             0             0            0              6              1
4             0             0            0              6              1


Define our complete feature list

In [ ]:
feature_columns = [
    # Sales history
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",

    # Rolling demand
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",

    # Demand trend
    "demand_ratio_7_28",
    "demand_change_7",

    # Price
    "sell_price",
    "price_change",
    "price_change_pct",
    "price_volatility_7",

    # Calendar
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "quarter",
    "year",
    "is_weekend",

    # Cyclical
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
    "week_sin",
    "week_cos",

    # Events
    "has_event_1",
    "has_event_2",
    "has_event",

    # SNAP
    "snap_CA",
    "snap_TX",
    "snap_WI",
    "snap_any",

    # Categorical IDs
    "item_id_code",
    "dept_id_code",
    "cat_id_code",
    "store_id_code",
    "state_id_code"
]

print("Number of features:", len(feature_columns))

Number of features: 40


Check the feature matrix

In [ ]:
feature_check = df[
    ["id", "date", "sales"] + feature_columns
]

feature_check.head()

,id,date,sales,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_14,rolling_mean_28,...,has_event,snap_CA,snap_TX,snap_WI,snap_any,item_id_code,dept_id_code,cat_id_code,store_id_code,state_id_code
0,FOODS_1_005_TX_3_validation,2011-01-29,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,6,1
1,FOODS_1_005_TX_3_validation,2011-01-30,3,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,6,1
2,FOODS_1_005_TX_3_validation,2011-01-31,3,3.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,6,1
3,FOODS_1_005_TX_3_validation,2011-02-01,2,3.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0,1,1,0,1,0,0,0,6,1
4,FOODS_1_005_TX_3_validation,2011-02-02,0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0,1,0,1,1,0,0,0,6,1


Check missing values

In [ ]:
missing_summary = (
    df[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary[
    missing_summary > 0
]

,0
price_volatility_7,140504
price_change,138704
price_change_pct,138704
sell_price,138404
lag_28,8400
rolling_mean_28,8400
demand_ratio_7_28,8400
demand_change_7,8400
rolling_std_28,8400
rolling_mean_14,4200


Remove rows without sufficient history

In [ ]:
model_df = df.dropna(
    subset=feature_columns
).copy()

In [ ]:
model_df = df.dropna(
    subset=feature_columns
).copy()

print("Original rows:", len(df))
print("Usable rows  :", len(model_df))
print(
    "Removed rows :",
    len(df) - len(model_df)
)

Original rows: 573900
Usable rows  : 431142
Removed rows : 142758


scaler must be fitted on training only

Import StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

Fit scaler ONLY on training data

In [ ]:
# Define split dates for training, validation, and test sets
train_end_date = pd.to_datetime('2015-05-24') # Example split point for training
val_end_date = pd.to_datetime('2016-03-24')   # Example split point for validation

# Split the data into training, validation, and test sets based on dates
train_features = model_df[model_df['date'] <= train_end_date]
validation_features = model_df[(model_df['date'] > train_end_date) & (model_df['date'] <= val_end_date)]
test_features = model_df[model_df['date'] > val_end_date]

print(f"Train data shape: {train_features.shape}")
print(f"Validation data shape: {validation_features.shape}")
print(f"Test data shape: {test_features.shape}")

X_train_raw = train_features[feature_columns]

X_val_raw = validation_features[feature_columns]

X_test_raw = test_features[feature_columns]

scaler.fit(X_train_raw)

print("Scaler fitted using TRAINING data only.")

Train data shape: (330675, 55)
Validation data shape: (91167, 55)
Test data shape: (9300, 55)
Scaler fitted using TRAINING data only.


Transform all three datasets

In [ ]:
X_train = scaler.transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

X_train: (330675, 40)
X_val  : (91167, 40)
X_test : (9300, 40)


Check scaling

In [ ]:
print(
    "Training feature means:",
    np.mean(X_train, axis=0)[:10]
)

print(
    "\nTraining feature std:",
    np.std(X_train, axis=0)[:10]
)

Training feature means: [-5.15703505e-18  6.87604674e-17  4.93356353e-17 -5.87901996e-17
  1.19643213e-16  7.28860954e-17  2.79855102e-16 -2.09719425e-17
  1.80668128e-16 -1.42677970e-16]

Training feature std: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


Save the scaler

In [ ]:
import joblib

scaler_path = f"{PROJECT_DIR}/models/feature_scaler.pkl"

joblib.dump(
    scaler,
    scaler_path
)

print("Scaler saved:")
print(scaler_path)

Scaler saved:
/content/drive/MyDrive/GC210_Retail_Forecasting/models/feature_scaler.pkl


Save feature data

In [ ]:
train_features.to_parquet(
    f"{PROCESSED_DIR}/train_features.parquet",
    index=False
)

validation_features.to_parquet(
    f"{PROCESSED_DIR}/validation_features.parquet",
    index=False
)

test_features.to_parquet(
    f"{PROCESSED_DIR}/test_features.parquet",
    index=False
)

print("Feature datasets saved successfully.")

Feature datasets saved successfully.


Save feature names

In [ ]:
import json

feature_info = {
    "num_features": len(feature_columns),
    "features": feature_columns
}

feature_path = f"{PROJECT_DIR}/feature_config.json"

with open(feature_path, "w") as f:
    json.dump(feature_info, f, indent=4)

print("Feature configuration saved.")

Feature configuration saved.


Final leakage check

In [ ]:
print("=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

print(
    "Training max date    :",
    train_features["date"].max()
)

print(
    "Validation min date  :",
    validation_features["date"].min()
)

print(
    "Validation max date  :",
    validation_features["date"].max()
)

print(
    "Test min date        :",
    test_features["date"].min()
)

print(
    "Test max date        :",
    test_features["date"].max()
)

print("\nScaler fitted on training data only: YES")
print("=" * 60)

LEAKAGE CHECK
Training max date    : 2015-05-24 00:00:00
Validation min date  : 2015-05-25 00:00:00
Validation max date  : 2016-03-24 00:00:00
Test min date        : 2016-03-25 00:00:00
Test max date        : 2016-04-24 00:00:00

Scaler fitted on training data only: YES
